# Unify IDs Notebook

> User-friendly by default, with fallback mode for maximum compatibility.

Run this notebook top-to-bottom. It supports:
- widget mode (`ipywidgets`) when available
- text fallback mode (`input(...)`) when widget support is missing

> Recommended setup (from repository root):
```bash
cd <path-to-your-clone>/awg-utils/unify_ids
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
pip install -r requirements-notebook.txt
```

> Then open `unify_ids.ipynb` and select the `.venv` kernel in VS Code/Jupyter.

If widgets are unavailable, the notebook will still work and ask for paths via plain text prompts.

## Enter input files
add the paths underneath the following cell:

In [2]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

json_path = ""
svg_folder = ""


def _normalize_user_path(raw_value):
    value = str(raw_value or "").strip().strip('"').strip("'")
    if not value:
        return ""
    value = os.path.expanduser(value)
    return os.path.abspath(value)


def _ensure_runtime_directory():
    cwd = Path.cwd()
    candidate_dirs = [cwd, cwd / "unify_ids", cwd.parent / "unify_ids"]
    for candidate in candidate_dirs:
        if (candidate / "unify_tkk_ids.py").is_file() and (candidate / "utils").is_dir():
            resolved = str(candidate.resolve())
            if str(cwd.resolve()) != resolved:
                os.chdir(resolved)
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            return resolved

    fallback = str(cwd.resolve())
    if fallback not in sys.path:
        sys.path.insert(0, fallback)
    return fallback


def _run_command(args):
    try:
        result = subprocess.run(
            args, capture_output=True, text=True, check=False, timeout=8
        )
    except Exception:
        return ""
    if result.returncode == 0:
        return result.stdout.strip()
    return ""


def _pick_first_installed(commands):
    for command in commands:
        if command and shutil.which(command[0]):
            return command
    return None


def _clipboard_command():
    if sys.platform == "darwin":
        return _pick_first_installed([["pbpaste"]])
    if sys.platform.startswith("linux"):
        return _pick_first_installed(
            [
                ["wl-paste", "-n"],
                ["xclip", "-selection", "clipboard", "-o"],
                ["xsel", "--clipboard", "--output"],
            ]
        )
    if os.name == "nt":
        return _pick_first_installed(
            [["powershell", "-NoProfile", "-Command", "Get-Clipboard"]]
        )
    return None


def _clipboard_text():
    command = _clipboard_command()
    if command is None:
        return ""
    return _run_command(command)


def _json_picker_commands():
    if sys.platform == "darwin":
        script = 'POSIX path of (choose file with prompt "Select JSON file")'
        return [["osascript", "-e", script]]
    if sys.platform.startswith("linux"):
        return [
            [
                "zenity",
                "--file-selection",
                "--title",
                "Select JSON file",
                "--file-filter=*.json",
            ],
            ["kdialog", "--getopenfilename", os.getcwd(), "*.json"],
        ]
    if os.name == "nt":
        ps = (
            "Add-Type -AssemblyName System.Windows.Forms;"
            "$d=New-Object System.Windows.Forms.OpenFileDialog;"
            "$d.Filter='JSON files (*.json)|*.json|All files (*.*)|*.*';"
            "if($d.ShowDialog() -eq [System.Windows.Forms.DialogResult]::OK){$d.FileName}"
        )
        return [["powershell", "-NoProfile", "-Command", ps]]
    return []


def _folder_picker_commands():
    if sys.platform == "darwin":
        script = 'POSIX path of (choose folder with prompt "Select SVG folder")'
        return [["osascript", "-e", script]]
    if sys.platform.startswith("linux"):
        return [
            [
                "zenity",
                "--file-selection",
                "--directory",
                "--title",
                "Select SVG folder",
            ],
            ["kdialog", "--getexistingdirectory", os.getcwd()],
        ]
    if os.name == "nt":
        ps = (
            "Add-Type -AssemblyName System.Windows.Forms;"
            "$d=New-Object System.Windows.Forms.FolderBrowserDialog;"
            "if($d.ShowDialog() -eq [System.Windows.Forms.DialogResult]::OK){$d.SelectedPath}"
        )
        return [["powershell", "-NoProfile", "-Command", ps]]
    return []


def _pick_with_commands(candidates):
    for command in candidates:
        if command and shutil.which(command[0]):
            value = _run_command(command)
            if value:
                return value
    return ""


def _choose_file_json():
    path = _normalize_user_path(_pick_with_commands(_json_picker_commands()))
    if path.lower().endswith(".json"):
        return path
    return ""


def _choose_folder():
    return _normalize_user_path(_pick_with_commands(_folder_picker_commands()))


runtime_dir = _ensure_runtime_directory()
print(f"Notebook runtime directory: {runtime_dir}")

try:
    import ipywidgets as widgets
    from IPython.display import display
    widgets_error = None
except Exception as exc:
    widgets = None
    display = None
    widgets_error = exc

if widgets is None:
    print("ipywidgets is not available. Using text-input fallback mode.")
    print("Optional UI install: pip install ipywidgets")
    print(f"Widget import error: {widgets_error}")
    json_path = _normalize_user_path(input("Path to .json file: "))
    svg_folder = _normalize_user_path(input("Path to SVG folder: "))
else:
    json_path_text = widgets.Text(
        description="JSON:",
        placeholder="Path to .json file",
        layout=widgets.Layout(width="62%"),
    )
    svg_folder_text = widgets.Text(
        description="SVG:",
        placeholder="Path to SVG folder",
        layout=widgets.Layout(width="62%"),
    )

    browse_json_button = widgets.Button(
        description="Browse",
        button_style="info",
        layout=widgets.Layout(width="15%"),
    )
    paste_json_button = widgets.Button(
        description="Paste from Clipboard",
        button_style="warning",
    )
    browse_svg_button = widgets.Button(
        description="Browse",
        button_style="info",
        layout=widgets.Layout(width="15%"),
    )
    paste_svg_button = widgets.Button(
        description="Paste from Clipboard",
        button_style="warning",
    )

    def _sync_paths(_=None):
        global json_path, svg_folder
        json_path = _normalize_user_path(json_path_text.value)
        svg_folder = _normalize_user_path(svg_folder_text.value)

    def on_browse_json(_):
        path = _choose_file_json()
        if path:
            json_path_text.value = path

    def on_paste_json(_):
        path = _normalize_user_path(_clipboard_text())
        if path:
            json_path_text.value = path

    def on_browse_svg(_):
        path = _choose_folder()
        if path:
            svg_folder_text.value = path

    def on_paste_svg(_):
        path = _normalize_user_path(_clipboard_text())
        if path:
            svg_folder_text.value = path

    can_pick_json = any(shutil.which(cmd[0]) for cmd in _json_picker_commands())
    can_pick_folder = any(shutil.which(cmd[0]) for cmd in _folder_picker_commands())
    can_clipboard = _clipboard_command() is not None

    browse_json_button.disabled = not can_pick_json
    browse_svg_button.disabled = not can_pick_folder
    paste_json_button.disabled = not can_clipboard
    paste_svg_button.disabled = not can_clipboard

    browse_json_button.on_click(on_browse_json)
    paste_json_button.on_click(on_paste_json)
    browse_svg_button.on_click(on_browse_svg)
    paste_svg_button.on_click(on_paste_svg)
    json_path_text.observe(_sync_paths, names="value")
    svg_folder_text.observe(_sync_paths, names="value")

    display(widgets.HBox([json_path_text, browse_json_button, paste_json_button]))
    display(widgets.HBox([svg_folder_text, browse_svg_button, paste_svg_button]))
    _sync_paths()

    notices = []
    if not can_pick_json or not can_pick_folder:
        notices.append(
            "Browse buttons are disabled because no supported file picker tool is available."
        )
    if not can_clipboard:
        notices.append(
            "Clipboard buttons are disabled because no supported clipboard tool is available."
        )
    for notice in notices:
        print(f"Compatibility note: {notice}")

    print("You can always type paths directly into the text boxes.")

Notebook runtime directory: /Users/Elias/awg-utils/unify_ids


You can always type paths directly into the text boxes.


## Run TKK Unifier

In [3]:
import os
import sys

# Reload modules to get the latest changes
if 'unify_tkk_ids' in sys.modules:
    del sys.modules['unify_tkk_ids']
if 'utils.extraction_utils' in sys.modules:
    del sys.modules['utils.extraction_utils']
if 'utils.svg_utils' in sys.modules:
    del sys.modules['utils.svg_utils']

from unify_tkk_ids import unify_tkk_ids

dry_run = False
verbose = True

# Validate paths
if not json_path or not os.path.isfile(json_path):
    print("Error: No valid JSON file selected")
elif not svg_folder or not os.path.isdir(svg_folder):
    print("Error: No valid SVG folder selected")
else:
    print("Running TKK Unifier...")
    print(f"  JSON: {json_path}")
    print(f"  SVG Folder: {svg_folder}")
    print(f"  dry_run={dry_run}, verbose={verbose}")
    result = unify_tkk_ids(
        json_path,
        svg_folder,
        dry_run=dry_run,
        verbose=verbose,
    )
    print(f"\n✓ Finished, result = {result}")


Running TKK Unifier...
  JSON: /Users/Elias/awg-utils/unify_tkk_ids/tests/data/textcritics.json
  SVG Folder: /Users/Elias/awg-utils/unify_tkk_ids/tests/img
  dry_run=False, verbose=True
--- Starting TKK ID processing ---
Loaded JSON with 25 entries
Found 30 SVG files in folder

Processing Entry ID: op25_WE
 Standard anchor: op25_WE
 Relevant SVGs (0): []
 No svgGroupIds to process

Processing Entry ID: M_317_TF1
 Standard anchor: M_317_TF1
 Relevant SVGs (2): ['M317_Textfassung1-1von2-final.svg', 'M317_Textfassung1-2von2-final.svg']
 [JSON] Changing 'g1145' -> 'g-tkk-m_317_tf1-1'
 [SVG] Changing 'g1145' -> 'g-tkk-m_317_tf1-1' in M317_Textfassung1-1von2-final.svg
 [JSON] Changing 'g1198' -> 'g-tkk-m_317_tf1-2'
 [SVG] Changing 'g1198' -> 'g-tkk-m_317_tf1-2' in M317_Textfassung1-1von2-final.svg
 [JSON] Changing 'g1204' -> 'g-tkk-m_317_tf1-3'
 [SVG] Changing 'g1204' -> 'g-tkk-m_317_tf1-3' in M317_Textfassung1-1von2-final.svg

Processing Entry ID: M_317_Sk1
 Standard anchor: M_317_Sk1
 Rel

## Run LinkBox Unifier
Unify LinkBox svgGroupIds with format `{entry_id}to{sheetId}`

In [6]:
import os
import sys

# Reload modules to get the latest changes
if 'unify_link_box_ids' in sys.modules:
    del sys.modules['unify_link_box_ids']
if 'utils.extraction_utils' in sys.modules:
    del sys.modules['utils.extraction_utils']
if 'utils.svg_utils' in sys.modules:
    del sys.modules['utils.svg_utils']

from unify_link_box_ids import unify_link_box_ids

# Validate paths
if not json_path or not os.path.isfile(json_path):
    print("❌ Error: No valid JSON file selected")
elif not svg_folder or not os.path.isdir(svg_folder):
    print("❌ Error: No valid SVG folder selected")
else:
    print(f'Running LinkBox Unifier...')
    print(f'  JSON: {json_path}')
    print(f'  SVG Folder: {svg_folder}')
    result = unify_link_box_ids(json_path, svg_folder)
    print(f'\n✓ Finished, result = {result}')

Running LinkBox Unifier...
  JSON: /Users/Elias/awg-utils/unify_tkk_ids/tests/data/textcritics.json
  SVG Folder: /Users/Elias/awg-utils/unify_tkk_ids/tests/img
--- Starting Link Box ID processing ---
Loaded JSON with 25 entries
Found 30 SVG files in folder

Processing Entry ID: op25_WE
 Standard anchor: op25_WE
 Relevant SVGs (0): []
 No linkBoxes to process

Processing Entry ID: M_317_TF1
 Standard anchor: M_317_TF1
 Relevant SVGs (2): ['M317_Textfassung1-1von2-final.svg', 'M317_Textfassung1-2von2-final.svg']
 No linkBoxes to process

Processing Entry ID: M_317_Sk1
 Standard anchor: M_317_Sk1
 Relevant SVGs (1): ['M317_Sk1-1von1-final.svg']
 Found 9 linkBox(es)
 [JSON] Changing 'g6407' -> 'g-lb-m_317_sk1-to-m_317_sk2'
 [SVG] Changing 'g6407' -> 'g-lb-m_317_sk1-to-m_317_sk2' in M317_Sk1-1von1-final.svg
 [JSON] Changing 'g6443' -> 'g-lb-m_317_sk1-to-m_317_sk2_1'
 [SVG] Changing 'g6443' -> 'g-lb-m_317_sk1-to-m_317_sk2_1' in M317_Sk1-1von1-final.svg
 [JSON] Changing 'g6490' -> 'g-lb-m_31